# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate Snowflake's incremental query behavior and SQL transformation logic across Silver (Iceberg) and Gold (Dynamic Tables).

In [ ]:
import duckdb
import pandas as pd

# Initialize DuckDB memory connection
con = duckdb.connect(database=':memory:', read_only=False)
print("DuckDB initialized for SQL simulation.")

In [ ]:
# 1. Simulate Silver Iceberg Tables (using DuckDB tables)
con.execute("""
    CREATE TABLE silver_condition_occurrence (
        person_id BIGINT,
        condition_concept_id INT,
        condition_start_date DATE
    );
    
    CREATE TABLE silver_drug_exposure (
        person_id BIGINT,
        drug_concept_id INT,
        drug_exposure_start_date DATE
    );
""")

# Insert Mock Data
con.execute("""
    INSERT INTO silver_condition_occurrence VALUES 
    (101, 32020, '2023-01-10'),
    (101, 31415, '2023-02-15'),
    (102, 32020, '2023-03-01');
    
    INSERT INTO silver_drug_exposure VALUES
    (101, 1908234, '2023-01-11'),
    (103, 1908234, '2023-04-10');
""")

print("Silver tables mocked.")

In [ ]:
# 2. Simulate dt_silver_patient_events (Snowflake Dynamic Table)
con.execute("""
    CREATE VIEW dt_silver_patient_events AS
    SELECT 
      person_id,
      condition_concept_id as event_concept_id,
      condition_start_date as event_date,
      'CONDITION' as event_type
    FROM silver_condition_occurrence
    UNION ALL
    SELECT 
      person_id,
      drug_concept_id as event_concept_id,
      drug_exposure_start_date as event_date,
      'DRUG' as event_type
    FROM silver_drug_exposure;
""")

print("dt_silver_patient_events:")
print(con.execute("SELECT * FROM dt_silver_patient_events ORDER BY person_id, event_date").df())

In [ ]:
# 3. Simulate dt_gold_patient_summary (Snowflake Dynamic Table)
con.execute("""
    CREATE VIEW dt_gold_patient_summary AS
    SELECT 
      person_id,
      COUNT(CASE WHEN event_type = 'CONDITION' THEN 1 END) as total_conditions,
      COUNT(CASE WHEN event_type = 'DRUG' THEN 1 END) as total_drugs,
      MAX(event_date) as last_event_date
    FROM dt_silver_patient_events
    GROUP BY person_id;
""")

print("\ndt_gold_patient_summary:")
print(con.execute("SELECT * FROM dt_gold_patient_summary ORDER BY person_id").df())

In [ ]:
# 4. Simulate incremental update behavior
con.execute("""
    INSERT INTO silver_condition_occurrence VALUES (102, 11111, '2023-03-15');
    INSERT INTO silver_drug_exposure VALUES (102, 22222, '2023-03-16');
""")

print("\ndt_gold_patient_summary after incremental Silver inserts:")
print(con.execute("SELECT * FROM dt_gold_patient_summary ORDER BY person_id").df())